In [ ]:
# Import necessary packages
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
import torch.nn as nn # Contains Required functions and layers
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from numpy.random import seed
from time import time
import os
from torch.utils.data import Dataset, DataLoader
seed(8)


In [ ]:
#read data, define data and one-hot encoded target values
data = pd.read_csv('semeion.csv', header=0, delimiter=',') #read data from semeion.csv file
c = data.iloc[:,:]
c = np.array(c)
z = torch.from_numpy(c) #convert data to numpy

#define train and test dataset
(train,test) = train_test_split(z,test_size=0.2,random_state=10) #train_test_split data (80% train, 20% test)
train_dataloader = DataLoader(train, batch_size=1, shuffle=True) #create data_loader
print(f'Created `torch_train_dataloader` with {len(train_dataloader)} batches!')

#creating train and test data
train_data = train[:,:256] #train data
train_targets = train[:,256:] #train targets
test_data = test[:,:256] #test data
test_targets = test[:,256:] #test labels
[mtr,ntr] = (train_data.shape) #size of train data
[mtest,ntest] = (test_data.shape) #size of test data
print(train_targets.shape)
print(test_targets.shape)

In [ ]:
import numpy as np

# Implement the ReLU activation function
def relu(x):
    return np.maximum(0, x)

def leaky_relu(x):
  neg = np.minimum(0,x)*0.015625
  pos = np.maximum(0,x)
  return neg + pos


In [ ]:
#final inference code
def do_it(data, testd, w12, w23, b12, b23):
    images = data[:, :256]  # Extract the image data from the input data
    labels = data[:, 256:].numpy()  # Extract the labels from the input data
    images = images.view(images.shape[0], -1).numpy()  # Reshape the image data
    labels_ts = np.zeros(testd)  # Create an array to store the target labels for the test data

    for i in range(testd):
        labels_ts[i] = np.argmax(labels[i, :])  # Convert the one-hot encoded labels to class indices
        labels_ts[i] = labels_ts[i] - 1  # Adjust the labels to be in the range [-1, 1]

    success = 0  # Initialize a variable to count the number of successful predictions
    for i in range(testd):
        a1 = images[i, :]
        h1 = np.dot(w12, a1) + np.transpose(b12)         
        h2 = np.transpose(relu(h1))
        # reduce bit precision of h2
        # h2 = h2 * 16
        # h2 = np.round(h2, decimals=0)  # Example: rounding to 0 decimal places
        # h2 = h2 / 2
        o1 = np.dot(w23, h2) + b23
        # reduce bit precision of o1
        # o1 = o1 * 8
        # o1 = np.round(o1, decimals=0)  # Example: rounding to 0 decimal places
        # o1 = o1 / 8
        o1 = relu(o1)
        out = np.argmax(o1)
        out = out - 1
        if out == labels_ts[i]:
            success += 1  # Increment the success count if the prediction matches the label
    print(success)
    acc = success * 100 / testd  # Calculate the accuracy as a percentage
    return acc  # Return the accuracy


In [ ]:
# extract parameter from w1_semeion.csv, b1_semeion.csv, w2_semeion.csv, b2_semeion.csv
import pandas as pd
import numpy as np
precision_value = 0
ww1 = pd.read_csv('w1_semeion.csv', header=0, delimiter=',') #read data from w1_semeion.csv file
ww1 = ww1.to_numpy()
ww1 = ww1 * 16
ww1 = np.round(ww1, decimals=precision_value)

bb1 = pd.read_csv('b1_semeion.csv', header=0, delimiter=',') #read data from b1_semeion.csv file
bb1 = bb1.to_numpy()
bb1 = bb1 * 16
bb1 = np.round(bb1, decimals=precision_value)

ww2 = pd.read_csv('w2_semeion.csv', header=0, delimiter=',') #read data from w2_semeion.csv file
ww2 = ww2.to_numpy()
ww2 = ww2 * 16
ww2 = np.round(ww2, decimals=precision_value)

bb2 = pd.read_csv('b2_semeion.csv', header=0, delimiter=',') #read data from b2_semeion.csv file
bb2 = bb2.to_numpy()
bb2 = bb2 * 16
bb2 = np.round(bb2, decimals=precision_value)
bb2 = bb2 * 16


print("extracted parameters")
#print(ww1)
#print(bb1)
#print(ww2)
#print(bb2)

In [ ]:

#get test accuracy
data = test
testd = mtest
print(do_it(data,testd,ww1,ww2,bb1,bb2))

#get train accuracy
data = train
testd = mtr
print(do_it(data,testd,ww1,ww2,bb1,bb2))

In [ ]:
# # store trained parameters as .mem files
# def to_signed_5bit(value):
#     if value < 0:
#         value = (1 << 5) + int(value)  # Convert to two's complement for negative values
#     return format(int(value), '05b')  # Format as a 5-bit binary string

# #store parameters as 7 bit binary
# def to_signed_7bit(value):
#     if value < 0:
#         value = (1 << 7) + int(value)  # Convert to two's complement for negative values
#     return format(int(value), '07b')  # Format as a 7-bit binary string

# def to_signed_6bit(value):
#     if value < 0:
#         value = (1 << 6) + int(value)  # Convert to two's complement for negative values
#     return format(int(value), '06b')  # Format as a 6-bit binary string

# with open('w1.mem', 'w') as f:
#     for row in w1:
#         for val in row:
#             f.write(to_signed_5bit(val) + ' ')
#         f.write('\n')

# with open('w2.mem', 'w') as f:
#     for row in w2:
#         for val in row:
#             f.write(to_signed_5bit(val) + ' ')
#         f.write('\n')

# with open('b1.mem', 'w') as f:
#     for val in b1:
#         f.write(to_signed_7bit(val) + '\n')

# bb2 = bb2 / 16
# with open('b2.mem', 'w') as f:
#     for val in b2:
#         f.write(to_signed_6bit(val) + '\n')
# bb2 = bb2 * 16
